# JEPA-Anything — minimal demo

Paper: [arXiv:2609.20800](https://arxiv.org/abs/2609.20800)  
Code: [Gen-Verse/JEPA-Anything](https://github.com/Gen-Verse/JEPA-Anything)  
Checkpoints: [HF collection](https://huggingface.co/collections/Gen-Verse/jepa-anything) / dataset `Gen-Verse/jepa-anything`

**Lightest domain checkpoint used here:** Synthetic Causal Dynamics (~1.9 MB).

Set `DRY_RUN = True` to validate imports/plans without downloading weights.

In [ ]:
DRY_RUN = True  # flip to False on a GPU Colab with network to download the 1.9 MB ckpt
CLONE_UPSTREAM = True
WORK = "/content/jepa-anything-work"  # change for local runs

import os, sys, subprocess, json
from pathlib import Path

work = Path(WORK)
work.mkdir(parents=True, exist_ok=True)
print("DRY_RUN=", DRY_RUN)
print("work=", work)

In [ ]:
def sh(cmd, cwd=None, check=True):
    print("+", cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

repo = work / "JEPA-Anything"
if CLONE_UPSTREAM:
    if not repo.exists():
        sh("git clone --depth 1 https://github.com/Gen-Verse/JEPA-Anything.git", cwd=work)
    core = repo / "jepa-anything-core"
    if core.exists():
        sh(f"{sys.executable} -m pip install -q -e '{core}[dev]'", check=False)
    else:
        print("WARN: jepa-anything-core missing — inspect upstream layout")
else:
    print("Skip clone")

In [ ]:
# Structural recipe (no trained weights / no paper metrics)
recipe = repo / "recipes" / "synthetic-linear-dynamics" / "run_recipe.py"
if recipe.exists():
    sh(f"{sys.executable} {recipe} --quiet", cwd=repo, check=False)
else:
    print("Recipe not found; continuing.")

In [ ]:
HF_REPO = "Gen-Verse/jepa-anything"
HF_FILE = (
    "05_Ten_Task_Dynamics/Synthetic_Causal_Dynamics/checkpoints/"
    "Synthetic_Causal_Dynamics_final_checkpoint.pt"
)
ckpt_dir = work / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

planned = f"python test/infer_dynamics.py --checkpoint {HF_FILE} --output /tmp/jepa_smoke/predictions.npz"
print("Planned infer command:")
print(planned)

if DRY_RUN:
    print("DRY_RUN: not downloading checkpoint.")
else:
    sh(f"{sys.executable} -m pip install -q huggingface_hub torch")
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=HF_FILE, local_dir=str(ckpt_dir))
    print("downloaded", path)
    import torch
    obj = torch.load(path, map_location="cpu", weights_only=False)
    print("loaded", type(obj), list(obj)[:20] if isinstance(obj, dict) else "")
    infer = repo / "test" / "infer_dynamics.py"
    if infer.exists():
        out = work / "predictions"
        out.mkdir(exist_ok=True)
        sh(f"{sys.executable} {infer} --checkpoint {path} --output {out}/predictions.npz", check=False)
    else:
        print("infer_dynamics.py missing in clone; torch.load smoke only.")

## GPU / RAM notes

- Structural recipe: CPU fine.
- Synthetic Causal Dynamics ckpt: ~1.9 MB; CPU torch.load is enough for a smoke test.
- Vision / Weather / Molecular domain checkpoints: often multi-GB + GPU — see each domain README on HF.
- Do not expect paper metrics from this notebook alone.